Performs a campaign to identify binding free energy and equilibrium concentrations for a list of candidate cDNA oligos against a specified aptamer.

Requirements:
- NUPACK installation (obtain from website)
- Pandas installation (should come with setting up NUPACK properly)

To run on Yale Bouchet cluster:
1. Create conda environment for nupack, installing all dependencies
2. Create a kernel accessing the nupack conda environment
3. Use a remote desktop to access jupyter lab
4. Run notebook from remote desktop, using nupack kernel

In [1]:
# Import NUPACK Python module
from nupack import *

In [14]:
# Read .csv file of cDNA sequences (must have a column labeled 'sequence')
import pandas as pd
df = pd.read_csv('cdna_sequences.csv', usecols=['sequence'])
seqs = df['sequence'].dropna().str.strip().str.upper().reset_index(drop=True)

# Define parameters
material = 'dna'
temperature = 25
na_conc = 0.325
mg_conc = 0.010
aptamer_starting_conc = 1e-7
cdna_starting_conc = 2e-7
aptamer_sequence = 'GCTTCCAGCTTATTGAATTACACGCAGAGGGTAGCGGCTCTGCGCATTCAATTGCTGCGCGCTGAAGCGCGGAAGC'

# Define physical model
my_model = Model(material=material, celsius=temperature, sodium=na_conc, magnesium=mg_conc)

# Define strand species
aptamer = Strand(aptamer_sequence, name='aptamer')

# Create tubes
tubes_list = []
for i, sequence in enumerate(seqs, start=1):
    cdna = Strand(sequence, name=f'cdna{i}')
    tube = Tube(strands={aptamer:aptamer_starting_conc, cdna:cdna_starting_conc}, 
                complexes=SetSpec(max_size=2, include=[[aptamer,cdna]], exclude=[[aptamer, aptamer], [cdna, cdna]]), 
                name=f't{i}')
    tubes_list.append(tube)

# Analyze tubes
tube_results = tube_analysis(tubes=tubes_list, model=my_model)

# Obtain delta G and concentration values for each tube and each complex
dG_cdna, dG_com = [], []
conc_apt, conc_cdna, conc_com = [], [], []

for i in range(len(seqs)):
    dG_cdna.append(tube_results[f'(cdna{i+1})'].free_energy)
    dG_com.append(tube_results[f'(aptamer+cdna{i+1})'].free_energy)
    
    for my_complex, conc in tube_results[f't{i+1}'].complex_concentrations.items():
        strands = [s.name for s in my_complex.strands]
        if len(strands) == 2:
            conc_com.append(conc)
        elif strands[0] == 'aptamer':
            conc_apt.append(conc)
        else:
            conc_cdna.append(conc)

dG_apt = [tube_results[f'(aptamer)'].free_energy]*len(seqs)

dG_bin = [a - b - c for a, b, c in zip(dG_com, dG_apt, dG_cdna)]
perc_apt_bound = [(1-n/aptamer_starting_conc)*100 for n in conc_apt]

results = {'cdna_seq': seqs,
           'deltaG_cdna': dG_cdna, 
           'deltaG_apt': dG_apt, 
           'deltaG_complex': dG_com,
           'deltaG_binding': dG_bin,
           'conc_apt': conc_apt,
           'conc_cdna': conc_cdna,
           'conc_complex': conc_com,
           '%_aptamer_bound': perc_apt_bound
          }

# Put results into csv file
df = pd.DataFrame(results)
df.to_csv("results.csv", index=False)